In [1]:
import os
import re
from openai import OpenAI
import fitz
import z_ocr_fn
import json

# PDF names

In [4]:
def file_name(folder):
    names = [os.path.splitext(f)[0] for f in os.listdir(folder) if f.endswith(".pdf") ]
    return names

# LLM FN

In [53]:
def names(text):
    prompt = f"""You are a data extarctor from messy OCR text from a CVs.Extract the following information from the OCR text:

1. Full candidate_name of the candidate
2. Phone number 

Rules:
- DO NOT give email details as candidate_name
- Return ONLY valid JSON
- Do not add explanations
- Do not guess information

Text:{text}
Output format:
{{
 "candidate_name": "",
 "candidate_phone": ""
}}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 200,
        response_format={"type": "json_object"}
    )
    output = response.choices[0].message.content.strip()

    return json.loads(output)

In [58]:
def one_json(y):
    text = z_ocr_fn.extraction(f"{y}.pdf")
    file = names(text)
    return file

In [70]:
def pdf_checking(file):
    doc = fitz.open(file)
    text = ""
    for i in doc:
        text = text + i.get_text()
    
    return len(text)

In [72]:
print(z_ocr_fn.extraction("y_bimali.pdf"))

CamScanner


# json files.

In [61]:
def json_file(output_folder):
    names_of_cv = file_name(output_folder)

    os.makedirs(output_folder,exist_ok=True)
    
    for i,x in enumerate(names_of_cv,start = 1):
        
        json_path = os.path.join(output_folder,f"{x}.json")
        if os.path.exists(json_path):
            continue

        llm_json = one_json(x)
        llm_json["candidate_num"] = i

        with open(f"cv_json_files\\{x}.json","w",encoding="utf-8") as f:
            json.dump(llm_json,f,indent = 4)

In [62]:
json_file("cv_files")